In [ ]:
%load_ext autoreload
%autoreload 2
import SepVector
from fwix import CudaOperator
from fwix import CudaWEM
from pyVector import superVector
import numpy as np
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt

In [ ]:
def ricker_wavelet(frequency, Nt, dt, t0=0):
    t = np.linspace(0, (Nt-1)*dt, Nt)
    t_shifted = t - t0
    ricker = (1 - 2 * (np.pi ** 2) * (frequency ** 2) * (t_shifted ** 2)) * np.exp(-(np.pi ** 2) * (frequency ** 2) * (t_shifted ** 2))
    return ricker

# Parameters
frequency = 10  # Ricker wavelet central frequency
Nt = 200    # Length of the wavelet in seconds
dt = 0.01      # Time sampling interval
t0 = 0.1

# Generate Ricker wavelet
wavelet = ricker_wavelet(frequency, Nt, dt, t0=t0)
W = np.fft.fft(wavelet)

df = 1/(Nt*dt)
of = 1.
maxf = 30.
nf = int((maxf-of) / df)

nf = 50
nz = 50

f, a = plt.subplots(1,2, figsize=[10,2])
a[0].plot(wavelet, label='Ricker Wavelet')
a[1].plot(np.abs(W[int(of/df):nf]), label="Spectrue")

In [ ]:
nf

In [ ]:
nx = 100
dx = 0.01
ny = 100
dy = 0.01
dz = 0.01

ns = 3;
ds = 1

nxrec = nx
nyrec = ny

slow = SepVector.getSepVector(ns=[nx,ny,nf,nz], ds=[dx, dy, df, dz], os=[0,   0,   of,  0], storage='dataComplex')
# constant
slow[:] = 1.
# add a layer
slow[nz-5:,...] = 2.

# 4 corners
# slow[:,:,:100,:100] = 1/1**2
# slow[:,:,:100,100:] = 1/2**2
# slow[:,:,100:,:100] = 1/3**2
# slow[:,:,100:,100:] = 1/4**2

# add attenuation in the borders
npml = 20
amax = 1e-1
pml = 1j * np.linspace(0,amax,npml) 

# slow[:,:,:npml,:] += -pml[::-1, np.newaxis] 
# slow[:,:,-npml:,:] += -pml[:, np.newaxis]
# slow[:,:,:,:npml] += -pml[::-1]
# slow[:,:,:,-npml:] += -pml

den = slow.clone()
den.set(1)

model = superVector(slow, den)

n_src_traces = ns
src_traces = SepVector.getSepVector(ns=[nf, n_src_traces],ds=[df, 1],os=[1, 0], storage='dataComplex')
n_rec_traces = nxrec * nyrec * ns
rec_traces = SepVector.getSepVector(ns=[nf, n_rec_traces],ds=[df, 1],os=[1, 0], storage='dataComplex')

In [ ]:
plt.plot(slow[:,0,0,0].real)

In [ ]:
n_rec_traces

In [ ]:
sx = np.random.uniform(0, (nx-1)*dx, n_src_traces).astype(np.float32) * 0 + (nx-1)*dx/2
sy = np.random.uniform(0, (ny-1)*dy, n_src_traces).astype(np.float32) * 0 + (ny-1)*dy/2
sz = np.random.uniform(0, (nz-1)*dz, n_src_traces).astype(np.float32) * 0 + dz
s_ids = np.linspace(0, ns-1, ns).astype(int)

rx, ry = np.meshgrid(np.linspace(0, (nx-1)*dx, nxrec), np.linspace(0, (ny-1)*dy, nyrec))
r_ids = s_ids.repeat(rx.size)
rx = np.tile(rx.flatten(), ns)
ry = np.tile(ry.flatten(), ns)
rz = np.zeros(rx.size) + (nz-1)*dz*0

In [ ]:
sz

In [ ]:
rz

In [ ]:
import genericIO
import os 

nbatches = [1, 1]
look = 2

par = {
    "nref" : 3,
    "eps" : 0.04,
    "padx" : nx,
    "pady" : ny,
    "taperx" : 0,
    "tapery" : 0,
    "ref_look_ahead" : look,
    "compress_rate" : 8.,
    "wflds_to_store" : 4,
    "wfld_path" : os.environ["SCRATCH"],
}
par = genericIO.pythonParams(par)

geometry = {
    "sx" : sx.tolist(), "sy" : sy.tolist(), "sz" : sz.tolist(), "s_ids" : s_ids.astype(np.int32).tolist(),
    "rx" : rx.tolist(), "ry" : ry.tolist(), "rz" : rz.tolist(), "r_ids" : r_ids.astype(np.int32).tolist(),
}

In [ ]:
src_traces

In [ ]:
par

In [ ]:

for i in range(n_src_traces):
    src_traces[i, :] = W[:nf]
    
propOp = CudaWEM.Propagator(
    model, rec_traces, src_traces, par, geometry)

In [ ]:
%%timeit -r 1
propOp.forward(False, model, rec_traces)

In [ ]:
rec_traces.norm()

In [ ]:
import holoviews as hv 
hv.extension('bokeh')
hv.output(widget_location='bottom')

In [ ]:
snaps = {}
every = 10
pclip = .001

amin = pclip * np.amin(rec_traces[:].real)
amax = pclip * np.amax(rec_traces[:].real)
# wfld_c = np.clip(wfld.real, amin, amax)

for i in range(nf):
    mat = rec_traces[:,i].real
    # only look at the recorded data from the first shot
    mat = mat[:nxrec*nyrec].reshape((nyrec, nxrec))
    snaps[i] = hv.Image(mat, bounds=(0,-nyrec,nxrec,0)).opts(cmap='gray', clim=(-pclip, pclip), aspect=nxrec//nyrec)

In [ ]:
hmap = hv.HoloMap(snaps, kdims='Frequency')
hmap

In [ ]:
hv.Curve(np.abs(rec_traces[2500,:])).opts(aspect=2)

In [ ]:
all_freq = np.zeros((rec_traces.shape[0], Nt), dtype=np.complex64)


all_freq[:,int(of/df):rec_traces.shape[1]+int(of/df)] = rec_traces[:]

time = np.fft.ifft(all_freq, axis=1).real

In [ ]:
snaps = {}
every = 10

cval = np.percentile(np.abs(time[:]), 100)

for i in range(0, Nt, every):
    mat = time[:,i].real
    mat = mat[:nxrec*nyrec].reshape((nyrec, nxrec))
    snaps[i] = hv.Image(mat, bounds=(0,-nyrec,nxrec,0)).opts(cmap='gray', clim=(-cval, cval), aspect=nxrec//nyrec)

In [ ]:
hmap = hv.HoloMap(snaps, kdims='Time')
hmap

In [ ]:
mat = time[int(nz/2-1)*nx:int(nz/2)*nx,:].real.T
mat = mat / np.amax(mat)
bounds = [0, -(Nt-1)*dt, (nx-1)*dx, 0]

hv.Image(mat, bounds=bounds).opts(cmap='gray', clim=(-1, 1), aspect=2)

In [ ]:
hv.Curve(mat[:,50]).opts(aspect=3, show_grid=True)

In [ ]:
bornOp = CudaWEM.ExtendedMigration(
    model, rec_traces, model, propOp)

In [ ]:
dmodel = model.clone()
dmodel.zero()
iz = int(nz/2)
dmodel[0][iz, :, int(ny/2), int(nx/2)] = 1.

In [ ]:
%%timeit -r 1
bornOp.migrate(False, dmodel[0], rec_traces)

In [ ]:
%%timeit -r 1
bornOp.forward(False, dmodel, rec_traces)

In [ ]:
rec_traces.norm()

In [ ]:
dmodel[0].norm()

In [ ]:
(Nt-1)*dt 

In [ ]:
snaps = {}
every = 10
pclip = .001

amin = pclip * np.amin(rec_traces[:].real)
amax = pclip * np.amax(rec_traces[:].real)
# wfld_c = np.clip(wfld.real, amin, amax)

for i in range(nf):
    mat = rec_traces[:,i].real
    # only look at the recorded data from the first shot
    mat = mat[:nxrec*nyrec].reshape((nyrec, nxrec))
    snaps[i] = hv.Image(mat, bounds=(0,-nyrec,nxrec,0)).opts(cmap='gray', clim=(-pclip, pclip), aspect=nxrec//nyrec)

In [ ]:
hmap = hv.HoloMap(snaps, kdims='Frequency')
hmap

In [ ]:
hv.Curve(np.abs(rec_traces[2500,:])).opts(aspect=2)

In [ ]:
all_freq = np.zeros((rec_traces.shape[0], Nt), dtype=np.complex64)


all_freq[:,int(of/df):rec_traces.shape[1]+int(of/df)] = rec_traces[:]

time = np.fft.ifft(all_freq, axis=1).real

In [ ]:
snaps = {}
every = 10
pclip = .001

amin = pclip * np.amin(time[:])
amax = pclip * np.amax(time[:])

for i in range(0, Nt, every):
    mat = time[:,i].real
    mat = mat[:nxrec*nyrec].reshape((nyrec, nxrec))
    snaps[i] = hv.Image(mat, bounds=(0,-nyrec,nxrec,0)).opts(cmap='gray', clim=(-pclip, pclip), aspect=nxrec//nyrec)

In [ ]:
hmap = hv.HoloMap(snaps, kdims='Time')
hmap

In [ ]:
mat = time[int(nz/2-1)*nx:int(nz/2)*nx,:].real.T
mat = mat / np.amax(mat)
bounds = [0, -(Nt-1)*dt, (nx-1)*dx, 0]

hv.Image(mat, bounds=bounds).opts(cmap='gray', clim=(-1, 1), aspect=2)

In [ ]:
hv.Curve(mat[:,50]).opts(aspect=3, show_grid=True)

In [ ]:
image = model.clone()

In [ ]:
%%timeit -r 1

bornOp.adjoint(False, image, rec_traces)

In [ ]:
image.norm()

In [ ]:
rec_traces.norm()

In [ ]:
snaps_zy = {}
snaps_zx = {}
every = 10
pclip = .0000001

amin = pclip * np.amin(dmodel[0][:].real)
amax = pclip * np.amax(dmodel[0][:].real)
# wfld_c = np.clip(wfld.real, amin, amax)

freq = 10

for i in range(0, nx, 10):
    mat_zy = dmodel[0][:,freq,:,i].real
    mat_zx = dmodel[0][:,freq,i,:].real
    snaps_zy[i] = hv.Image(mat_zy, bounds=(-nz,0,0,nx)).opts(cmap='gray', clim=(-pclip, pclip), aspect=1)

In [ ]:
hmap = hv.HoloMap(snaps_zy, kdims='x')
hmap

In [ ]:
snaps_zy = {}
snaps_zx = {}
every = 10
cval = np.percentile(np.abs(dmodel[0][3:,...]), 99.5)

for i in range(0, nx, 10):
    mat_zy = np.mean(dmodel[0][3:,:,:,i].real, axis=1)
    mat_zx = np.mean(dmodel[0][3:,:,i,:].real, axis=1)
    snaps_zy[i] = hv.Image(mat_zy, bounds=(-nz,0,0,nx)).opts(cmap='gray', clim=(-cval, cval), aspect=1)

In [ ]:
hmap = hv.HoloMap(snaps_zy, kdims='x')
hmap

In [ ]:
hv.Curve(np.mean(image[0][:,:,50,50].real, axis=1))

In [ ]:
snaps_zy = {}
snaps_zx = {}
every = 10

cval = np.percentile(np.abs(image[0][:,...]), 90)

for i in range(0, nz, 2):
    mat_zy = np.mean(image[0][i,:,:,:].real, axis=0)
    snaps_zy[i] = hv.Image(mat_zy, bounds=(-ny,0,0,nx)).opts(cmap='gray', clim=(-cval, cval), aspect=1)

In [ ]:
hmap = hv.HoloMap(snaps_zy, kdims='x')
hmap